# Temporal Climate Metrics: Diurnal Temperature Range & Degree Hours

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ByMaxAnjos/LCZ4py/blob/main/notebooks/local/06_temporal_climate_metrics.en.ipynb)

**Learning objective**: learn to compute two classic temporal climate indicators —
Diurnal Temperature Range (DTR) and Cooling/Heating Degree Hours (CDH/HDH) — from
station air-temperature records, and to aggregate both by Local Climate Zone (LCZ)
class using `lcz_dtr` and `lcz_degree_hours` from **LCZ4py**. We use the Berlin
station dataset (`lcz_data_berlin.csv`) throughout, consistent with the rest of the
`local/` tutorial series.

## Why DTR and Degree Hours matter for urban climate

The **Diurnal Temperature Range (DTR)** — the difference between a station's daily
maximum and minimum air temperature — is one of the oldest and most robust
fingerprints of urbanization in the climate record. Stewart & Oke's Local Climate
Zone (LCZ) scheme (Stewart & Oke, 2012, *Bulletin of the American Meteorological
Society*) was created precisely because "urban vs. rural" is too coarse a binary
to explain why some built-up sites warm at night while others do not: compact
high-rise cores (LCZ 1–3), open low-rise suburbs (LCZ 6), and industrial zones
(LCZ 8) all have very different sky view factors, thermal admittance, and
anthropogenic heat release, and therefore very different diurnal temperature
cycles even though they are all "urban."

Mechanistically, dense built LCZs tend to **suppress DTR** relative to vegetated
or water-dominated LCZs (11–17): high-thermal-admittance construction materials
(concrete, asphalt) absorb and store solar radiation during the day, and a low
sky view factor (tall buildings, street canyons) traps outgoing longwave
radiation at night, slowing radiative cooling. The net effect is a muted daily
minimum — daytime maxima are similar to or only slightly higher than rural sites,
but nighttime minima stay markedly warmer. This asymmetry (small daytime warming,
large nighttime warming) is one of the classic mechanistic explanations for the
**nighttime Urban Heat Island (UHI)**, and comparing DTR across LCZ classes with
the *same* station network is a direct, data-driven test of this mechanism —
no separate "urban" and "rural" labels needed, just the LCZ class each station
sits in.

**Degree Hours** — Cooling Degree Hours (CDH) and Heating Degree Hours (HDH) —
are the standard energy-demand proxy used across building energy simulation,
utility load forecasting, and public-health heat-exposure studies. Relative to a
comfort/balance-point temperature (`base_temp`, commonly 18°C in European and US
practice), CDH accumulates how far and how long the air temperature exceeds that
threshold (cooling/AC demand, and heat-exposure risk), while HDH accumulates how
far and how long it falls below it (heating demand). Because both are computed
**per hour** and then summed per day, they capture both the *intensity* and the
*duration* of thermal stress — something a simple daily mean temperature cannot.

Stratifying CDH/HDH by LCZ class turns this energy-demand proxy into an urban-form
diagnostic: if compact/impervious LCZs (1–3, 8–10) show systematically higher CDH
than open/vegetated LCZs (6, 9, 11–17) at the *same* ambient conditions, that is
direct evidence that urban form — not just regional climate — drives cooling
energy burden and heat exposure. This is exactly the kind of evidence that feeds
climate adaptation and heat action planning: it identifies which LCZ classes are
the priority targets for cool-roof retrofits, urban greening, or shading
interventions. See Anjos et al. (2025, *Scientific Reports*,
https://www.nature.com/articles/s41598-025-92000-0) for the LCZ4r methodology
this Python port (LCZ4py) implements, and Stewart & Oke (2012) for the original
LCZ classification scheme.

## Install LCZ4py

In [1]:
!pip -q install "LCZ4py[all] @ git+https://github.com/ByMaxAnjos/LCZ4py.git"


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: /Applications/QGIS.app/Contents/MacOS/bin/python3 -m pip install --upgrade pip
ERROR: Package 'lcz4py' requires a different Python: 3.9.5 not in '>=3.10'


## Setup: LCZ map and station data

We reuse the same Berlin LCZ map and station CSV as the rest of the `local/`
series: `lcz_get_map(city="Berlin")` for the LCZ raster, and
`lcz_data_berlin.csv` for the hourly air-temperature observations
(`var="airT"`, `station_id="station"`).

In [2]:
import pandas as pd
from LCZ4py import lcz_get_map
from LCZ4py.local.lcz_temporal import lcz_dtr, lcz_degree_hours

lcz_map_path = lcz_get_map(city="Berlin")
df = pd.read_csv("https://raw.githubusercontent.com/ByMaxAnjos/LCZ4py/main/lcz_data_berlin.csv")
df.head()

06:24:25 - LCZ4py._internal._lcz_map_engine - INFO - Loading clipped map from local cache.


,Unnamed: 0,date,station,airT,Latitude,Longitude
0,806404,2019-01-01 00:00:00,airportthf,8.50000,52.467500,13.402100
1,806405,2019-01-01 00:00:00,airporttxl,7.90000,52.564400,13.308800
2,806406,2019-01-01 00:00:00,albrecht,8.04725,52.444594,13.348607
3,806407,2019-01-01 00:00:00,bamberger,8.27166,52.496494,13.337552
4,806408,2019-01-01 00:00:00,baruth,8.20000,52.061300,13.499700


## `lcz_dtr` — Diurnal Temperature Range by LCZ class

`lcz_dtr(x, data_frame, var, station_id, sp_res=100.0, tp_res="day", by=None, lang="en", year=None, month=None, day=None, hour=None, start=None, end=None)`
computes, for **each station and each calendar day**, `DTR = max(T) − min(T)`
from the hourly (or sub-daily) observations, then assigns every station to its
LCZ class via the LCZ raster `x` (nearest-pixel lookup at the station's
lat/lon).

Key parameters:
- `x` — the LCZ raster (path or `rasterio` dataset), used only to classify
  stations into LCZ 1–17.
- `data_frame`, `var`, `station_id` — the observation table and the columns
  holding the temperature variable and station identifier.
- `sp_res` — spatial resolution in metres; informational here (no gridding is
  performed, so it does not affect the result).
- `tp_res` — temporal resolution, currently `"day"` (DTR is inherently a daily
  quantity).
- `by` — optional temporal grouping (e.g. `"season"`) for faceting.
- `year`/`month`/`day`/`hour`/`start`/`end` — the usual date/time filters
  shared across `local/` functions.

It returns a `DTRResult` dataclass with:
- `df` — a Polars DataFrame, one row per station per day, columns
  `station, lcz, date, dtr, t_max, t_min, lcz_name`.
- `plot` — a Plotly box plot of DTR distributions grouped by LCZ class, with
  an annotation separating urban (LCZ 1–10) from natural/water classes
  (LCZ 11–17).

**What to look for**: if the urban LCZ boxes (left side) sit systematically
*lower* than the natural LCZ boxes (right side), that is the suppressed-DTR
signature described above — evidence of a nighttime-UHI mechanism operating
in this exact station network.

In [3]:
dtr_result = lcz_dtr(
    lcz_map_path,
    data_frame=df,
    var="airT",
    station_id="station",
    lang="en",
)
dtr_result.df.head()

station,lcz,date,t_max,t_min,dtr,lcz_name
str,i32,date,f64,f64,f64,str
"""airportthf""",14,2019-01-01,8.6,3.6,5.0,"""Low plants"""
"""airportthf""",14,2019-01-02,3.4,-1.5,4.9,"""Low plants"""
"""airportthf""",14,2019-01-03,1.9,-2.7,4.6,"""Low plants"""
"""airportthf""",14,2019-01-04,4.9,-0.5,5.4,"""Low plants"""
"""airportthf""",14,2019-01-05,8.0,2.2,5.8,"""Low plants"""


In [4]:
dtr_result.plot

### Interpreting the DTR output

Look at the median line and interquartile range of each LCZ's box: compact/
built-up classes typically cluster at a lower DTR than open, low-rise, or
natural classes. Wide boxes (large IQR) for a given LCZ suggest strong
day-to-day variability — e.g. driven by cloud cover or wind — while a
LCZ-to-LCZ gap in the medians is the structural, urban-form-driven signal we
are after. The `df` table lets you drill into individual station-days (e.g.
to find the days with the most extreme urban-rural DTR contrast).

## `lcz_degree_hours` — Cooling/Heating Degree Hours by LCZ class

`lcz_degree_hours(x, data_frame, var, station_id, base_temp=18.0, degree_type="cooling", sp_res=100.0, by=None, lang="en", year=None, month=None, day=None, hour=None, start=None, end=None)`
computes hourly degree-hours relative to `base_temp` and accumulates them per
station per day:

- **Cooling** (`degree_type="cooling"`): `CDH = Σ max(0, T − base_temp)` per
  hour, summed per day — proxy for air-conditioning demand / heat exposure.
- **Heating** (`degree_type="heating"`): `HDH = Σ max(0, base_temp − T)` per
  hour, summed per day — proxy for space-heating demand.

Key parameters:
- `base_temp` — the comfort/balance-point threshold in °C (default 18.0, the
  common European/US convention).
- `degree_type` — `"cooling"` or `"heating"`.
- Remaining parameters (`x`, `data_frame`, `var`, `station_id`, `sp_res`, `by`,
  date filters) mirror `lcz_dtr`.

It returns a `DegreeHoursResult` dataclass with:
- `df` — per-station daily accumulated degree hours, columns
  `station, lcz, date, degree_hours, lcz_name`.
- `total` — aggregated per LCZ class, columns `lcz, lcz_name, mean, std, total`.
- `plot` — a Plotly bar chart of **total** degree hours per LCZ class (error
  bars = std across station-days), again annotated urban vs. natural.

We run it twice below — once for cooling, once for heating — to see both ends
of the thermal-comfort spectrum for the same station network.

In [5]:
cdh_result = lcz_degree_hours(
    lcz_map_path,
    data_frame=df,
    var="airT",
    station_id="station",
    base_temp=18.0,
    degree_type="cooling",
    lang="en",
)
cdh_result.total

lcz,mean,std,total,lcz_name
i32,f64,f64,f64,str
2,27.36,51.24,19153.44,"""Compact midrise"""
4,29.62,55.49,20730.75,"""Open highrise"""
5,27.06,50.29,18941.78,"""Open midrise"""
6,24.86,46.4,86998.05,"""Open lowrise"""
11,23.87,45.57,50131.59,"""Dense trees"""
12,24.14,46.05,33797.22,"""Scattered trees"""
14,26.39,50.27,36941.81,"""Low plants"""


In [6]:
cdh_result.plot

In [7]:
hdh_result = lcz_degree_hours(
    lcz_map_path,
    data_frame=df,
    var="airT",
    station_id="station",
    base_temp=18.0,
    degree_type="heating",
    lang="en",
)
hdh_result.total

lcz,mean,std,total,lcz_name
i32,f64,f64,f64,str
2,156.52,130.12,109566.5,"""Compact midrise"""
4,155.11,131.22,108573.93,"""Open highrise"""
5,173.05,134.26,121133.4,"""Open midrise"""
6,177.29,132.8,620501.72,"""Open lowrise"""
11,181.53,134.12,381202.56,"""Dense trees"""
12,185.95,132.98,260333.31,"""Scattered trees"""
14,169.15,133.09,236807.43,"""Low plants"""


In [8]:
hdh_result.plot

### Interpreting the Degree Hours output

In the cooling run, LCZ classes with the tallest bars are those accumulating
the most hours above the comfort threshold — these are the priority targets
for cooling-energy-demand reduction (shading, cool roofs, greening) and for
heat-health early-warning outreach. In the heating run, the pattern often
(though not always) inverts: open/vegetated or high-elevation classes can
accumulate more heating degree hours if they run cooler on average. Comparing
the two side by side tells a climate-adaptation planner which LCZ classes
are "cooling-dominated" (compact urban), which are "heating-dominated," and
which are balanced — directly informing where to prioritize adaptation
investment. The `total` table is the compact summary to export into a report;
`df` gives you the full daily series per station for deeper time-series work
(e.g. combining with `lcz_ts` from notebook 1).

## Conclusion & next steps

In this notebook we computed two temporal climate metrics — Diurnal
Temperature Range and Cooling/Heating Degree Hours — from Berlin's station
network and aggregated both by LCZ class using `lcz_dtr` and
`lcz_degree_hours`. DTR gave us a mechanistic window into nighttime UHI
(suppressed diurnal range in compact urban LCZs), while CDH/HDH translated
temperature into an energy-demand and heat-exposure proxy, stratified by
urban form.

**Next**: [`07_thermal_comfort_anthropogenic_heat`](07_thermal_comfort_anthropogenic_heat.en.ipynb)
builds directly on this foundation — it computes the Universal Thermal
Climate Index (UTCI) for human heat-stress assessment and estimates
anthropogenic heat flux (Q_F) per LCZ class, extending the energy-balance
story started here with degree hours.